# 📄 Gestion des Documents avec Métadonnées par PJ - AskMe Search

Ce notebook permet de gérer l'indexation des documents avec **métadonnées individuelles par pièce jointe**.

**Architecture :**
- 📁 **Document** = Dossier avec ID unique (36 caractères)
- 📎 **Pièce jointe** = Fichier avec ses propres métadonnées
- 🔐 **Droits** = Au niveau de chaque PJ individuellement
- 📊 **Métadonnées** = Par fichier dans metadata.json

**Use cases :**
- 📁 Préparer la structure client
- 🔄 Synchronisation incrémentale (PJ modifiées uniquement)
- 📋 Lister et analyser les PJ individuellement
- 🔍 Recherche avec métadonnées par PJ
- 🗑️ Suppression au niveau PJ
- 📊 Gestion des métadonnées évolutives par fichier

## 📦 Configuration et Imports

In [ ]:
import sys
sys.path.append('../scripts')

from simple_indexer import SimpleIndexer, create_simple_index
from pathlib import Path
import json
import hashlib
import os
import tempfile
import requests
from typing import List, Dict, Optional
from datetime import datetime
import time

# Configuration
OPENSEARCH_URL = "http://localhost:9200"
CLIENTS_DATA_DIR = "../clients-data"

print("📄 Notebook Gestion Documents Multi-PJ chargé")
print(f"   OpenSearch: {OPENSEARCH_URL}")
print(f"   Données clients: {CLIENTS_DATA_DIR}")
print(f"   Architecture: Document (36 chars) + PJ multiples")

## 🏢 Configuration Client

In [ ]:
# Choisir le client à gérer
CLIENT_ID = "entreprise-demo"  # Changez selon votre client

# Chemins dérivés
client_dir = Path(CLIENTS_DATA_DIR) / CLIENT_ID
documents_dir = client_dir / "documents"
metadata_file = client_dir / "metadata.json"
index_state_file = client_dir / ".index_state.json"

# Index OpenSearch pour ce client
index_name = f"askme-{CLIENT_ID}"
indexer = SimpleIndexer(opensearch_url=OPENSEARCH_URL, index_name=index_name)

print(f"🏢 Client configuré: {CLIENT_ID}")
print(f"   📁 Répertoire: {client_dir}")
print(f"   📄 Documents: {documents_dir}")
print(f"   📊 Métadonnées: {metadata_file}")
print(f"   🔍 Index OpenSearch: {index_name}")

# Vérifier l'existence des répertoires
if client_dir.exists():
    stats = indexer.get_index_stats()
    if stats:
        print(f"   ✅ Client existant: {stats.get('documents_count', 0)} chunks indexés")
    else:
        print(f"   ⚠️ Client existant mais index vide")
else:
    print(f"   ❌ Client inexistant - Utilisez prepare_client_structure() pour le créer")

## 🏗️ Préparation Structure Client

In [ ]:
def prepare_client_structure(client_id: str, sample_documents: Dict[str, Dict] = None) -> bool:
    """
    Préparer la structure complète d'un client avec métadonnées par PJ
    
    Args:
        client_id: ID du client
        sample_documents: {
            "document-uuid": {
                "rapport.pdf": {
                    "securityRights": ["finance", "direction"],
                    "titreDocument": "Rapport Financier Q4 2024",
                    "fieldMetadata": "auteur:Marie Dupont;departement:Finance;version:1.0",
                    "metadata_storage_name": "rapport.pdf",
                    "metadata_storage_path": "/documents/uuid/rapport.pdf"
                }
            }
        }
    """
    
    print(f"🏗️ Préparation structure client: {client_id}")
    
    try:
        # Créer l'arborescence
        client_path = Path(CLIENTS_DATA_DIR) / client_id
        documents_path = client_path / "documents"
        
        client_path.mkdir(parents=True, exist_ok=True)
        documents_path.mkdir(exist_ok=True)
        
        print(f"   ✅ Répertoires créés")
        
        # Créer l'index OpenSearch
        index_created = create_simple_index(OPENSEARCH_URL, f"askme-{client_id}")
        if index_created:
            print(f"   ✅ Index OpenSearch créé: askme-{client_id}")
        
        # Initialiser l'état d'indexation
        state_path = client_path / ".index_state.json"
        with open(state_path, 'w', encoding='utf-8') as f:
            json.dump({}, f, indent=2)
        
        print(f"   ✅ État d'indexation initialisé")
        
        # Créer des dossiers de test si demandé
        if sample_documents:
            print(f"\n📁 Création de dossiers de test:")
            for doc_id, pj_metadata in sample_documents.items():
                doc_dir = documents_path / doc_id
                doc_dir.mkdir(exist_ok=True)
                
                # Créer le fichier metadata.json pour ce document
                metadata_file = doc_dir / "metadata.json"
                with open(metadata_file, 'w', encoding='utf-8') as f:
                    json.dump(pj_metadata, f, indent=2, ensure_ascii=False)
                
                print(f"   📂 {doc_id}/ ({len(pj_metadata)} PJ)")
        
        print(f"\n✅ Structure client '{client_id}' prête !")
        print(f"💡 Placez vos documents dans: {documents_path}/[document-uuid]/")
        print(f"💡 Métadonnées par PJ dans: [document-uuid]/metadata.json")
        
        return True
        
    except Exception as e:
        print(f"❌ Erreur préparation: {e}")
        return False

def create_sample_documents_structure() -> Dict:
    """Créer une structure d'exemple avec les vrais champs métadonnées"""
    
    return {
        "123e4567-e89b-12d3-a456-426614174000": {
            "rapport_principal.pdf": {
                "securityRights": ["finance", "direction"],
                "titreDocument": "Rapport Financier Q4 2024",
                "fieldMetadata": "auteur:Marie Dupont;departement:Finance;date_creation:2024-01-15;version:1.0;type:principal",
                "metadata_storage_name": "rapport_principal.pdf",
                "metadata_storage_path": "/documents/123e4567-e89b-12d3-a456-426614174000/rapport_principal.pdf"
            },
            "annexe_financiere.xlsx": {
                "securityRights": ["finance"],
                "titreDocument": "Données Financières Détaillées",
                "fieldMetadata": "auteur:Jean Martin;departement:Finance;date_creation:2024-01-15;version:1.0;type:annexe",
                "metadata_storage_name": "annexe_financiere.xlsx",
                "metadata_storage_path": "/documents/123e4567-e89b-12d3-a456-426614174000/annexe_financiere.xlsx"
            },
            "presentation.pptx": {
                "securityRights": ["direction"],
                "titreDocument": "Présentation Comité Direction",
                "fieldMetadata": "auteur:Marie Dupont;departement:Direction;date_creation:2024-01-16;version:1.0;type:presentation",
                "metadata_storage_name": "presentation.pptx",
                "metadata_storage_path": "/documents/123e4567-e89b-12d3-a456-426614174000/presentation.pptx"
            }
        },
        "987fcdeb-51a2-43d1-9c4b-123456789abc": {
            "manuel_complet.pdf": {
                "securityRights": None,  # Document public
                "titreDocument": "Manuel Utilisateur v2.1",
                "fieldMetadata": "auteur:Équipe IT;departement:IT;date_creation:2024-01-10;version:2.1;type:manuel",
                "metadata_storage_name": "manuel_complet.pdf",
                "metadata_storage_path": "/documents/987fcdeb-51a2-43d1-9c4b-123456789abc/manuel_complet.pdf"
            },
            "guide_rapide.pdf": {
                "securityRights": None,  # Document public
                "titreDocument": "Guide de Démarrage Rapide",
                "fieldMetadata": "auteur:Équipe IT;departement:IT;date_creation:2024-01-10;version:2.1;type:guide",
                "metadata_storage_name": "guide_rapide.pdf",
                "metadata_storage_path": "/documents/987fcdeb-51a2-43d1-9c4b-123456789abc/guide_rapide.pdf"
            }
        }
    }

# Exemple de préparation
print(f"🏗️ Préparation de structure client:")
print("=" * 60)

if not client_dir.exists():
    print(f"⚠️ Client '{CLIENT_ID}' n'existe pas")
    print(f"⚠️ Décommentez pour créer la structure:")
    print()
    
    # DÉCOMMENTEZ POUR CRÉER LA STRUCTURE:
    # sample_structure = create_sample_documents_structure()
    # success = prepare_client_structure(CLIENT_ID, sample_structure)
    
    print(f"💡 Utilisation:")
    print(f"   prepare_client_structure('{CLIENT_ID}')  # Structure vide")
    print(f"   prepare_client_structure('{CLIENT_ID}', sample_structure)  # Avec exemples")
else:
    print(f"✅ Client '{CLIENT_ID}' existe déjà")
    
    # Afficher la structure existante
    print(f"📁 Documents existants:")
    doc_count = 0
    pj_count = 0
    
    for doc_dir in documents_dir.iterdir():
        if doc_dir.is_dir() and len(doc_dir.name) >= 32:
            doc_count += 1
            metadata_file = doc_dir / "metadata.json"
            
            if metadata_file.exists():
                try:
                    with open(metadata_file, 'r', encoding='utf-8') as f:
                        pj_metadata = json.load(f)
                    
                    pj_count += len(pj_metadata)
                    print(f"   📂 {doc_dir.name[:8]}... ({len(pj_metadata)} PJ)")
                    
                    for pj_name, metadata in list(pj_metadata.items())[:2]:  # Limiter à 2
                        titre = metadata.get('titreDocument', 'Sans titre')
                        field_meta = metadata.get('fieldMetadata', '')
                        rights = metadata.get('securityRights', None)
                        rights_str = f" 🔐" if rights else " 🌐"
                        print(f"      📄 {titre}{rights_str}")
                        if field_meta:
                            # Afficher quelques champs du fieldMetadata
                            fields = field_meta.split(';')[:2]  # Prendre 2 premiers
                            print(f"         {'; '.join(fields)}")
                    
                    if len(pj_metadata) > 2:
                        print(f"      ... et {len(pj_metadata) - 2} autres PJ")
                        
                except Exception as e:
                    print(f"   ⚠️ Erreur lecture métadonnées: {e}")
            else:
                # Compter les fichiers physiques
                files = [f for f in doc_dir.rglob('*') if f.is_file() and f.name != 'metadata.json']
                pj_count += len(files)
                print(f"   📂 {doc_dir.name[:8]}... ({len(files)} fichiers sans métadonnées)")
    
    print(f"\n📊 Total: {doc_count} documents, {pj_count} pièces jointes")

## 🔄 Système de Synchronisation Incrémentale

In [ ]:
def calculate_file_hash(file_path: Path) -> str:
    """Calculer le hash SHA256 d'un fichier"""
    hash_sha256 = hashlib.sha256()
    try:
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                hash_sha256.update(chunk)
        return hash_sha256.hexdigest()
    except Exception:
        return "error"

def calculate_document_signature(client_id: str, doc_id: str, doc_dir: Path) -> str:
    """Calculer signature unique d'un document (toutes PJ + métadonnées)"""
    
    # 1. Hash de toutes les PJ
    attachments_hash = []
    for pj_file in doc_dir.rglob('*'):
        if pj_file.is_file():
            file_hash = calculate_file_hash(pj_file)
            file_size = pj_file.stat().st_size
            file_mtime = pj_file.stat().st_mtime
            attachments_hash.append(f"{pj_file.name}:{file_hash}:{file_size}:{file_mtime}")
    
    # 2. Hash des métadonnées du document
    metadata_hash = get_document_metadata_hash(client_id, doc_id)
    
    # 3. Signature combinée
    combined = "|".join(sorted(attachments_hash)) + f"|meta:{metadata_hash}"
    return hashlib.sha256(combined.encode()).hexdigest()

def get_document_metadata_hash(client_id: str, doc_id: str) -> str:
    """Calculer le hash des métadonnées d'un document"""
    try:
        client_path = Path(CLIENTS_DATA_DIR) / client_id
        metadata_file = client_path / "metadata.json"
        
        if metadata_file.exists():
            with open(metadata_file, 'r', encoding='utf-8') as f:
                all_metadata = json.load(f)
            
            doc_metadata = all_metadata.get(doc_id, {})
            metadata_str = json.dumps(doc_metadata, sort_keys=True)
            return hashlib.md5(metadata_str.encode()).hexdigest()
        
        return "no_metadata"
    except Exception:
        return "error"

def load_index_state(state_file: Path) -> Dict:
    """Charger l'état d'indexation précédent"""
    try:
        if state_file.exists():
            with open(state_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        return {}
    except Exception:
        return {}

def save_index_state(state_file: Path, new_state: Dict) -> bool:
    """Sauvegarder le nouvel état d'indexation"""
    try:
        with open(state_file, 'w', encoding='utf-8') as f:
            json.dump(new_state, f, indent=2, ensure_ascii=False)
        return True
    except Exception as e:
        print(f"❌ Erreur sauvegarde état: {e}")
        return False

def detect_document_changes(client_id: str) -> Dict:
    """Détecter les changements au niveau document (avec toutes ses PJ)"""
    
    client_path = Path(CLIENTS_DATA_DIR) / client_id
    documents_path = client_path / "documents"
    state_file = client_path / ".index_state.json"
    
    if not documents_path.exists():
        return {'error': f'Répertoire documents inexistant: {documents_path}'}
    
    current_state = load_index_state(state_file)
    
    changes = {
        'new_documents': [],
        'modified_documents': [],
        'deleted_documents': [],
        'unchanged_documents': []
    }
    
    # Scanner chaque document (dossier UUID)
    current_docs = set()
    
    for doc_dir in documents_path.iterdir():
        if doc_dir.is_dir() and len(doc_dir.name) >= 32:  # UUID-like 
            doc_id = doc_dir.name
            current_docs.add(doc_id)
            
            # Compter les PJ
            attachments = [f for f in doc_dir.rglob('*') if f.is_file()]
            
            if not attachments:
                # Dossier vide, ignorer
                continue
            
            # Calculer signature du document
            doc_signature = calculate_document_signature(client_id, doc_id, doc_dir)
            
            if doc_id not in current_state:
                # Nouveau document
                changes['new_documents'].append({
                    'doc_id': doc_id,
                    'doc_dir': doc_dir,
                    'signature': doc_signature,
                    'attachments_count': len(attachments)
                })
            else:
                old_signature = current_state[doc_id].get('signature', '')
                
                if old_signature != doc_signature:
                    # Document modifié
                    changes['modified_documents'].append({
                        'doc_id': doc_id,
                        'doc_dir': doc_dir,
                        'signature': doc_signature,
                        'old_signature': old_signature,
                        'attachments_count': len(attachments)
                    })
                else:
                    # Document inchangé
                    changes['unchanged_documents'].append(doc_id)
    
    # Documents supprimés
    for doc_id in current_state:
        if doc_id not in current_docs:
            changes['deleted_documents'].append(doc_id)
    
    return changes

print(f"🔄 Fonctions de synchronisation chargées")
print(f"   ✅ Détection de changements par signature")
print(f"   ✅ Support des documents multi-PJ")
print(f"   ✅ Gestion de l'état incrémental")

## 📁 Indexation Documents Multi-PJ

In [ ]:
def get_pj_metadata(client_id: str, doc_id: str, pj_name: str) -> Dict:
    """Récupérer les métadonnées d'une pièce jointe spécifique"""
    try:
        client_path = Path(CLIENTS_DATA_DIR) / client_id
        doc_path = client_path / "documents" / doc_id
        metadata_file = doc_path / "metadata.json"
        
        if metadata_file.exists():
            with open(metadata_file, 'r', encoding='utf-8') as f:
                all_pj_metadata = json.load(f)
            return all_pj_metadata.get(pj_name, {})
        
        return {}
    except Exception as e:
        print(f"⚠️ Erreur lecture métadonnées {pj_name}: {e}")
        return {}

def index_document_with_attachments(client_id: str, doc_id: str, doc_dir: Path) -> bool:
    """Indexer un document avec les métadonnées exactes de ton système"""
    
    print(f"📄 Indexation document: {doc_id}")
    
    indexer_client = SimpleIndexer(opensearch_url=OPENSEARCH_URL, index_name=f"askme-{client_id}")
    
    indexed_count = 0
    processed_attachments = 0
    
    # Indexer chaque pièce jointe avec tes champs métadonnées exacts
    for pj_file in doc_dir.rglob('*'):
        if pj_file.is_file() and pj_file.name != 'metadata.json':
            pj_name = pj_file.name
            processed_attachments += 1
            
            print(f"   📎 PJ {processed_attachments}: {pj_name}")
            
            # Récupérer les métadonnées spécifiques de cette PJ
            pj_metadata = get_pj_metadata(client_id, doc_id, pj_name)
            
            # Extraction des champs selon ton image
            titre_document = pj_metadata.get('titreDocument', pj_name)
            field_metadata = pj_metadata.get('fieldMetadata', '')
            security_rights = pj_metadata.get('securityRights', None)
            metadata_storage_name = pj_metadata.get('metadata_storage_name', pj_name)
            metadata_storage_path = pj_metadata.get('metadata_storage_path', f'/documents/{doc_id}/{pj_name}')
            
            print(f"      📋 Titre: {titre_document}")
            print(f"      📊 FieldMetadata: {field_metadata}")
            if security_rights:
                rights_str = ", ".join(security_rights)
                print(f"      🔐 Droits: {rights_str}")
            else:
                print(f"      🌐 Accès libre")
            
            try:
                # Extraire contenu de la PJ
                content = indexer_client.processor.extract_text(pj_file)
                if not content or len(content.strip()) < 10:
                    print(f"      ⚠️ Contenu vide ou trop court")
                    continue
                
                # Découper en chunks
                chunks = indexer_client.splitter.split_text(content)
                if not chunks:
                    print(f"      ⚠️ Aucun chunk généré")
                    continue
                
                # Indexer chaque chunk avec TES CHAMPS EXACTS
                for i, chunk in enumerate(chunks):
                    # Génération des IDs selon ton système
                    chunk_id = f"{doc_id}_{pj_name}_chunk_{i}"
                    parent_id = doc_id
                    
                    # Document avec tes champs exacts de l'image
                    doc = {
                        # Champs obligatoires de ton image
                        "chunk_id": chunk_id,
                        "parent_id": parent_id,
                        "chunk": chunk,  # Le contenu du chunk
                        "title": titre_document,
                        
                        # Métadonnées de stockage
                        "metadata_storage_last_modified": datetime.now().isoformat(),
                        "metadata_storage_name": metadata_storage_name,
                        "metadata_storage_path": metadata_storage_path,
                        
                        # Droits de sécurité (StringCollection)
                        "securityRights": security_rights if security_rights else [],
                        
                        # Titre du document
                        "titreDocument": titre_document,
                        
                        # Métadonnées de champ (String avec format key:value;key:value)
                        "fieldMetadata": field_metadata
                    }
                    
                    # Note: text_vector sera généré par OpenSearch si configuré
                    
                    # Indexer le chunk
                    response = requests.post(
                        f"{indexer_client.opensearch_url}/{indexer_client.index_name}/_doc/{chunk_id}",
                        headers={"Content-Type": "application/json"},
                        json=doc
                    )
                    
                    if response.status_code in [200, 201]:
                        indexed_count += 1
                    else:
                        print(f"      ❌ Erreur indexation chunk {i}: {response.status_code}")
                
                print(f"      ✅ {len(chunks)} chunks indexés")
                
            except Exception as e:
                print(f"      ❌ Erreur PJ {pj_name}: {e}")
                continue
    
    print(f"   📊 Résultat: {indexed_count} chunks au total, {processed_attachments} PJ traitées")
    return indexed_count > 0

def create_test_document_structure(client_id: str) -> str:
    """Créer un document de test avec tes champs métadonnées exactes"""
    
    import uuid
    
    # Générer un UUID pour le document
    doc_id = str(uuid.uuid4())
    
    # Créer le dossier document
    client_path = Path(CLIENTS_DATA_DIR) / client_id
    doc_dir = client_path / "documents" / doc_id
    doc_dir.mkdir(parents=True, exist_ok=True)
    
    # Créer des PJ de test
    test_files = {
        "rapport_activite.txt": """
Rapport d'Activité Q4 2024
==========================

Ce document présente les résultats du quatrième trimestre 2024.

Résultats Financiers:
- Chiffre d'affaires: 2.5M€ (+15% vs Q3)
- Bénéfice net: 450K€ 
- Investissements: 300K€

Objectifs 2025:
- Croissance ciblée: +25%
- Nouveau marché: Europe du Nord
""",
        "specs_systeme.txt": """
Spécifications Techniques v2.0
===============================

Architecture Système:
- Infrastructure cloud: AWS
- Base de données: PostgreSQL 14
- Framework: React + Node.js

Performances:
- Temps de réponse: <200ms
- Disponibilité: 99.9%
- Throughput: 1000 req/s
""",
        "presentation_board.txt": """
Présentation Board - Stratégie 2025
===================================

Points Stratégiques:
✓ Expansion internationale
✓ Innovation produit
✓ Acquisitions ciblées

Budget prévu: 5M€
Horizon: 18 mois
"""
    }
    
    # Écrire les fichiers
    for filename, content in test_files.items():
        file_path = doc_dir / filename
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)
    
    # Créer les métadonnées avec TES CHAMPS EXACTS
    pj_metadata = {
        "rapport_activite.txt": {
            "securityRights": ["finance", "direction"],
            "titreDocument": "Rapport d'Activité Q4 2024",
            "fieldMetadata": "auteur:Marie Dupont;departement:Finance;date_creation:2024-01-15;version:1.0;type:rapport",
            "metadata_storage_name": "rapport_activite.txt",
            "metadata_storage_path": f"/documents/{doc_id}/rapport_activite.txt"
        },
        "specs_systeme.txt": {
            "securityRights": ["it", "tech"],
            "titreDocument": "Spécifications Techniques v2.0",
            "fieldMetadata": "auteur:Jean Martin;departement:IT;date_creation:2024-01-16;version:2.0;type:specification",
            "metadata_storage_name": "specs_systeme.txt",
            "metadata_storage_path": f"/documents/{doc_id}/specs_systeme.txt"
        },
        "presentation_board.txt": {
            "securityRights": ["direction", "board"],
            "titreDocument": "Présentation Board - Stratégie 2025",
            "fieldMetadata": "auteur:CEO Office;departement:Direction;date_creation:2024-01-17;version:1.0;type:presentation",
            "metadata_storage_name": "presentation_board.txt",
            "metadata_storage_path": f"/documents/{doc_id}/presentation_board.txt"
        }
    }
    
    # Sauvegarder les métadonnées par PJ
    metadata_file = doc_dir / "metadata.json"
    with open(metadata_file, 'w', encoding='utf-8') as f:
        json.dump(pj_metadata, f, indent=2, ensure_ascii=False)
    
    return doc_id

# Test d'indexation avec tes champs exacts
print(f"📁 Test d'indexation avec tes champs métadonnées:")
print("=" * 60)

if client_dir.exists():
    print(f"✅ Client {CLIENT_ID} disponible pour test")
    print(f"⚠️ Décommentez pour créer et indexer un document de test:")
    print()
    
    # DÉCOMMENTEZ POUR TESTER:
    # print(f"🏗️ Création document de test...")
    # test_doc_id = create_test_document_structure(CLIENT_ID)
    # print(f"   ✅ Document créé: {test_doc_id}")
    # print(f"   📁 Dossier: {documents_dir / test_doc_id}")
    # 
    # print(f"\n📄 Indexation du document...")
    # doc_dir = documents_dir / test_doc_id
    # success = index_document_with_attachments(CLIENT_ID, test_doc_id, doc_dir)
    # 
    # if success:
    #     print(f"\n🎉 Test réussi ! Document indexé avec tes champs exacts")
    # else:
    #     print(f"\n❌ Échec du test d'indexation")
    
    print(f"💡 Utilisation:")
    print(f"   test_doc_id = create_test_document_structure('{CLIENT_ID}')")
    print(f"   index_document_with_attachments('{CLIENT_ID}', test_doc_id, doc_dir)")
    print(f"\n📋 Champs utilisés (exactement comme ton image):")
    print(f"   • chunk_id, parent_id, chunk, title")
    print(f"   • metadata_storage_last_modified, metadata_storage_name, metadata_storage_path")
    print(f"   • securityRights, titreDocument, fieldMetadata")
else:
    print(f"❌ Client {CLIENT_ID} inexistant")
    print(f"💡 Utilisez d'abord prepare_client_structure()")

## 🔄 Synchronisation Incrémentale

In [ ]:
def delete_document_by_id(client_id: str, doc_id: str) -> bool:
    """Supprimer tous les chunks d'un document (toutes PJ)"""
    
    indexer_client = SimpleIndexer(opensearch_url=OPENSEARCH_URL, index_name=f"askme-{client_id}")
    
    # Rechercher tous les chunks de ce document
    search_body = {
        "query": {"term": {"document_id": doc_id}},
        "size": 1000
    }
    
    try:
        response = requests.post(
            f"{indexer_client.opensearch_url}/{indexer_client.index_name}/_search",
            headers={"Content-Type": "application/json"},
            json=search_body
        )
        
        if response.status_code == 200:
            hits = response.json().get('hits', {}).get('hits', [])
            deleted_count = 0
            
            for hit in hits:
                chunk_id = hit['_id']
                del_response = requests.delete(
                    f"{indexer_client.opensearch_url}/{indexer_client.index_name}/_doc/{chunk_id}"
                )
                if del_response.status_code == 200:
                    deleted_count += 1
            
            return deleted_count > 0
        
        return False
    except Exception as e:
        print(f"❌ Erreur suppression document {doc_id}: {e}")
        return False

def sync_client_incremental(client_id: str) -> Dict:
    """Synchronisation incrémentale d'un client"""
    
    print(f"🔄 Synchronisation incrémentale: {client_id}")
    print("=" * 60)
    
    # Détecter les changements
    changes = detect_document_changes(client_id)
    
    if 'error' in changes:
        print(f"❌ {changes['error']}")
        return {'error': changes['error']}
    
    sync_report = {
        'client_id': client_id,
        'timestamp': datetime.now().isoformat(),
        'processed': 0,
        'errors': [],
        'operations': []
    }
    
    # Résumé des changements
    total_changes = (len(changes['new_documents']) + 
                    len(changes['modified_documents']) + 
                    len(changes['deleted_documents']))
    
    print(f"📊 Changements détectés:")
    print(f"   🆕 Nouveaux: {len(changes['new_documents'])}")
    print(f"   🔄 Modifiés: {len(changes['modified_documents'])}")
    print(f"   🗑️ Supprimés: {len(changes['deleted_documents'])}")
    print(f"   ✅ Inchangés: {len(changes['unchanged_documents'])}")
    
    if total_changes == 0:
        print(f"\n✅ Aucun changement - Synchronisation à jour !")
        return sync_report
    
    print(f"\n🔄 Traitement de {total_changes} changements:")
    
    # 1. Supprimer les documents effacés
    for doc_id in changes['deleted_documents']:
        try:
            print(f"\n🗑️ Suppression: {doc_id[:8]}...")
            success = delete_document_by_id(client_id, doc_id)
            if success:
                sync_report['operations'].append(f"DELETED: {doc_id}")
                sync_report['processed'] += 1
                print(f"   ✅ Supprimé")
            else:
                print(f"   ⚠️ Aucun chunk à supprimer")
        except Exception as e:
            error_msg = f"Error deleting {doc_id}: {e}"
            sync_report['errors'].append(error_msg)
            print(f"   ❌ Erreur: {e}")
    
    # 2. Indexer nouveaux documents
    for doc_info in changes['new_documents']:
        try:
            doc_id = doc_info['doc_id']
            doc_dir = doc_info['doc_dir']
            
            print(f"\n🆕 Nouveau: {doc_id[:8]}... ({doc_info['attachments_count']} PJ)")
            success = index_document_with_attachments(client_id, doc_id, doc_dir)
            
            if success:
                sync_report['operations'].append(f"ADDED: {doc_id}")
                sync_report['processed'] += 1
            else:
                sync_report['errors'].append(f"Failed to add {doc_id}")
        except Exception as e:
            error_msg = f"Error adding {doc_info['doc_id']}: {e}"
            sync_report['errors'].append(error_msg)
            print(f"   ❌ Erreur: {e}")
    
    # 3. Réindexer documents modifiés
    for doc_info in changes['modified_documents']:
        try:
            doc_id = doc_info['doc_id']
            doc_dir = doc_info['doc_dir']
            
            print(f"\n🔄 Modifié: {doc_id[:8]}... ({doc_info['attachments_count']} PJ)")
            
            # Supprimer ancienne version
            delete_success = delete_document_by_id(client_id, doc_id)
            
            # Réindexer nouvelle version
            success = index_document_with_attachments(client_id, doc_id, doc_dir)
            
            if success:
                sync_report['operations'].append(f"UPDATED: {doc_id}")
                sync_report['processed'] += 1
            else:
                sync_report['errors'].append(f"Failed to update {doc_id}")
        except Exception as e:
            error_msg = f"Error updating {doc_info['doc_id']}: {e}"
            sync_report['errors'].append(error_msg)
            print(f"   ❌ Erreur: {e}")
    
    # 4. Sauvegarder nouvel état
    print(f"\n💾 Sauvegarde de l'état...")
    new_state = {}
    
    client_path = Path(CLIENTS_DATA_DIR) / client_id
    documents_path = client_path / "documents"
    
    for doc_dir in documents_path.iterdir():
        if doc_dir.is_dir() and len(doc_dir.name) >= 32:
            doc_id = doc_dir.name
            attachments = [f for f in doc_dir.rglob('*') if f.is_file()]
            
            if attachments:  # Ignorer les dossiers vides
                signature = calculate_document_signature(client_id, doc_id, doc_dir)
                new_state[doc_id] = {
                    'signature': signature,
                    'last_sync': datetime.now().isoformat(),
                    'attachments_count': len(attachments)
                }
    
    state_file = client_path / ".index_state.json"
    save_success = save_index_state(state_file, new_state)
    
    if save_success:
        print(f"   ✅ État sauvegardé")
    else:
        print(f"   ⚠️ Erreur sauvegarde état")
    
    # Résumé final
    print(f"\n📊 RÉSULTATS DE SYNCHRONISATION:")
    print("=" * 50)
    print(f"✅ Opérations réussies: {sync_report['processed']}")
    print(f"❌ Erreurs: {len(sync_report['errors'])}")
    
    if sync_report['operations']:
        print(f"\n📋 Opérations effectuées:")
        for op in sync_report['operations']:
            print(f"   • {op}")
    
    if sync_report['errors']:
        print(f"\n⚠️ Erreurs rencontrées:")
        for error in sync_report['errors']:
            print(f"   • {error}")
    
    return sync_report

# Test de synchronisation
print(f"🔄 Test de synchronisation incrémentale:")
print("=" * 60)

if client_dir.exists():
    print(f"✅ Client {CLIENT_ID} disponible")
    print(f"⚠️ Décommentez pour lancer la synchronisation:")
    print()
    
    # DÉCOMMENTEZ POUR SYNCHRONISER:
    # sync_report = sync_client_incremental(CLIENT_ID)
    
    print(f"💡 Utilisation:")
    print(f"   sync_report = sync_client_incremental('{CLIENT_ID}')")
    print(f"\n💡 Pour automatiser:")
    print(f"   # Script cron quotidien")
    print(f"   # python sync_all_clients.py")
else:
    print(f"❌ Client {CLIENT_ID} inexistant")
    print(f"💡 Créez d'abord la structure avec prepare_client_structure()")

## 🔍 Recherche avec Regroupement par Document

In [ ]:
def search_with_document_context(client_id: str, query: str, user_rights: List[str] = None, 
                               size: int = 10) -> Dict:
    """Recherche avec contexte document et regroupement par document"""
    
    indexer_client = SimpleIndexer(opensearch_url=OPENSEARCH_URL, index_name=f"askme-{client_id}")
    
    # Rechercher avec plus de résultats pour regrouper
    results = indexer_client.search(query, user_rights, size=size*5)  
    
    if not results or 'hits' not in results:
        return {
            'query': query,
            'user_rights': user_rights,
            'documents': [],
            'total_documents': 0,
            'total_chunks': 0
        }
    
    # Regrouper par document
    documents = {}
    total_chunks = 0
    
    for hit in results['hits']['hits']:
        source = hit['_source']
        doc_id = source.get('document_id', 'unknown')
        total_chunks += 1
        
        if doc_id not in documents:
            documents[doc_id] = {
                'document_id': doc_id,
                'document_title': source.get('document_title', 'Sans titre'),
                'document_metadata': source.get('document_metadata', {}),
                'access_rights': source.get('accessRights', None),
                'max_score': hit['_score'],
                'attachments': {},
                'total_chunks': 0,
                'best_matches': []
            }
        
        # Grouper par PJ dans le document
        attachment_name = source.get('attachment_name', 'unknown')
        
        if attachment_name not in documents[doc_id]['attachments']:
            documents[doc_id]['attachments'][attachment_name] = {
                'attachment_name': attachment_name,
                'attachment_type': source.get('attachment_type', ''),
                'attachment_description': source.get('attachment_description', ''),
                'chunks': [],
                'max_score': hit['_score']
            }
        
        # Ajouter ce chunk
        chunk_info = {
            'content': source.get('content', ''),
            'chunk_id': source.get('chunk_id', 0),
            'score': hit['_score'],
            'highlight': hit.get('highlight', {})
        }
        
        documents[doc_id]['attachments'][attachment_name]['chunks'].append(chunk_info)
        documents[doc_id]['total_chunks'] += 1
        
        # Mettre à jour les scores max
        documents[doc_id]['max_score'] = max(documents[doc_id]['max_score'], hit['_score'])
        documents[doc_id]['attachments'][attachment_name]['max_score'] = max(
            documents[doc_id]['attachments'][attachment_name]['max_score'], 
            hit['_score']
        )
        
        # Garder les meilleurs extraits
        if len(documents[doc_id]['best_matches']) < 3:
            documents[doc_id]['best_matches'].append({
                'attachment': attachment_name,
                'content': source.get('content', '')[:200] + '...',
                'score': hit['_score'],
                'highlight': hit.get('highlight', {})
            })
    
    # Trier par score et limiter
    sorted_docs = sorted(documents.values(), key=lambda x: x['max_score'], reverse=True)[:size]
    
    return {
        'query': query,
        'user_rights': user_rights,
        'documents': sorted_docs,
        'total_documents': len(sorted_docs),
        'total_chunks': total_chunks
    }

def display_search_results_grouped(search_results: Dict):
    """Afficher les résultats groupés par document"""
    
    query = search_results.get('query', '')
    user_rights = search_results.get('user_rights', None)
    documents = search_results.get('documents', [])
    total_docs = search_results.get('total_documents', 0)
    total_chunks = search_results.get('total_chunks', 0)
    
    print(f"🔍 Recherche: '{query}'")
    if user_rights:
        rights_str = ", ".join(user_rights)
        print(f"👤 Droits utilisateur: {rights_str}")
    else:
        print(f"🌐 Recherche libre")
    
    print(f"📊 {total_docs} documents trouvés ({total_chunks} chunks au total)")
    print("=" * 70)
    
    if not documents:
        print(f"❌ Aucun résultat")
        return
    
    for i, doc in enumerate(documents, 1):
        # En-tête document
        rights_icon = "🔐" if doc['access_rights'] else "🌐"
        print(f"{i}. {rights_icon} {doc['document_title']} (score: {doc['max_score']:.2f})")
        print(f"   📋 ID: {doc['document_id'][:8]}...")
        
        # Métadonnées document
        if doc['document_metadata']:
            metadata = doc['document_metadata']
            if 'auteur' in metadata:
                print(f"   👤 Auteur: {metadata['auteur']}")
            if 'departement' in metadata:
                print(f"   🏢 Département: {metadata['departement']}")
            if 'date_creation' in metadata:
                print(f"   📅 Date: {metadata['date_creation']}")
        
        # Droits d'accès
        if doc['access_rights']:
            rights_str = ", ".join(doc['access_rights'])
            print(f"   🔑 Droits: {rights_str}")
        else:
            print(f"   🌐 Accès libre")
        
        # Pièces jointes trouvées
        attachments = doc['attachments']
        print(f"   📎 {len(attachments)} PJ contiennent le terme ({doc['total_chunks']} chunks):")
        
        for att_name, att_info in attachments.items():
            att_type = att_info['attachment_type']
            att_desc = att_info['attachment_description']
            chunks_count = len(att_info['chunks'])
            
            type_icon = {
                'principal': '📄',
                'annexe': '📊', 
                'presentation': '📽️',
                'resume': '📋',
                'guide': '📖'
            }.get(att_type, '📎')
            
            print(f"      {type_icon} {att_name} ({chunks_count} extraits)")
            if att_desc:
                print(f"         💬 {att_desc}")
        
        # Meilleurs extraits
        print(f"   💡 Meilleurs extraits:")
        for j, match in enumerate(doc['best_matches'][:2], 1):
            content = match['content']
            if 'highlight' in match and 'content' in match['highlight']:
                content = match['highlight']['content'][0]
            
            print(f"      {j}. [{match['attachment']}] {content}")
        
        print()

# Test de recherche
print(f"🔍 Test de recherche avec regroupement:")
print("=" * 60)

if client_dir.exists():
    # Vérifier s'il y a des documents indexés
    stats = indexer.get_index_stats()
    if stats and stats.get('documents_count', 0) > 0:
        print(f"✅ Index contient {stats['documents_count']} chunks")
        print(f"⚠️ Décommentez pour tester la recherche:")
        print()
        
        # DÉCOMMENTEZ POUR TESTER LA RECHERCHE:
        # test_queries = ["rapport", "technique", "financier", "système"]
        # 
        # for query in test_queries[:2]:  # Tester 2 requêtes
        #     print(f"\n" + "="*70)
        #     results = search_with_document_context(CLIENT_ID, query, size=3)
        #     display_search_results_grouped(results)
        # 
        # # Test avec droits
        # print(f"\n" + "="*70)
        # print(f"🔐 Test avec droits ['finance']:")
        # results_with_rights = search_with_document_context(
        #     CLIENT_ID, "rapport", 
        #     user_rights=["finance"], 
        #     size=3
        # )
        # display_search_results_grouped(results_with_rights)
        
        print(f"💡 Utilisation:")
        print(f"   results = search_with_document_context('{CLIENT_ID}', 'query')")
        print(f"   display_search_results_grouped(results)")
    else:
        print(f"⚠️ Index vide - Indexez d'abord des documents")
        print(f"💡 Utilisez sync_client_incremental() ou créez des documents de test")
else:
    print(f"❌ Client {CLIENT_ID} inexistant")
    print(f"💡 Créez d'abord la structure avec prepare_client_structure()")

## 📊 Gestion des Métadonnées

In [ ]:
def update_document_metadata(client_id: str, doc_id: str, new_metadata: Dict) -> Dict:
    """Mettre à jour les métadonnées d'un document"""
    
    client_path = Path(CLIENTS_DATA_DIR) / client_id
    metadata_file = client_path / "metadata.json"
    
    try:
        # Charger métadonnées existantes
        if metadata_file.exists():
            with open(metadata_file, 'r', encoding='utf-8') as f:
                all_metadata = json.load(f)
        else:
            all_metadata = {}
        
        # Sauvegarder anciennes métadonnées pour comparaison
        old_metadata = all_metadata.get(doc_id, {})
        
        # Mettre à jour
        all_metadata[doc_id] = new_metadata
        
        # Sauvegarder
        with open(metadata_file, 'w', encoding='utf-8') as f:
            json.dump(all_metadata, f, indent=2, ensure_ascii=False)
        
        return {
            'status': 'success',
            'doc_id': doc_id,
            'message': f'Métadonnées mises à jour pour {doc_id}',
            'old_metadata': old_metadata,
            'new_metadata': new_metadata,
            'requires_reindex': True
        }
    
    except Exception as e:
        return {
            'status': 'error',
            'doc_id': doc_id,
            'message': f'Erreur mise à jour métadonnées: {e}'
        }

def list_documents_with_metadata(client_id: str) -> List[Dict]:
    """Lister tous les documents avec leurs métadonnées"""
    
    client_path = Path(CLIENTS_DATA_DIR) / client_id
    metadata_file = client_path / "metadata.json"
    documents_path = client_path / "documents"
    
    if not documents_path.exists():
        return []
    
    # Charger métadonnées
    metadata = {}
    if metadata_file.exists():
        try:
            with open(metadata_file, 'r', encoding='utf-8') as f:
                metadata = json.load(f)
        except Exception as e:
            print(f"⚠️ Erreur lecture métadonnées: {e}")
    
    documents_list = []
    
    # Scanner les dossiers documents
    for doc_dir in documents_path.iterdir():
        if doc_dir.is_dir() and len(doc_dir.name) >= 32:
            doc_id = doc_dir.name
            
            # Compter les PJ
            attachments = [f for f in doc_dir.rglob('*') if f.is_file()]
            
            if attachments:  # Ignorer dossiers vides
                doc_metadata = metadata.get(doc_id, {})
                
                documents_list.append({
                    'doc_id': doc_id,
                    'doc_dir': str(doc_dir),
                    'attachments_count': len(attachments),
                    'attachments_list': [f.name for f in attachments],
                    'metadata': doc_metadata,
                    'title': doc_metadata.get('titreDocument', doc_id[:8] + '...'),
                    'security_rights': doc_metadata.get('securityRights', None),
                    'field_metadata': doc_metadata.get('fieldMetadata', {}),
                    'has_metadata': len(doc_metadata) > 0
                })
    
    return documents_list

def analyze_metadata_distribution(client_id: str) -> Dict:
    """Analyser la répartition des métadonnées"""
    
    documents = list_documents_with_metadata(client_id)
    
    analysis = {
        'total_documents': len(documents),
        'with_metadata': 0,
        'without_metadata': 0,
        'with_security_rights': 0,
        'security_rights_distribution': {},
        'field_metadata_keys': {},
        'attachment_types': {}
    }
    
    for doc in documents:
        if doc['has_metadata']:
            analysis['with_metadata'] += 1
        else:
            analysis['without_metadata'] += 1
        
        # Droits de sécurité
        if doc['security_rights']:
            analysis['with_security_rights'] += 1
            for right in doc['security_rights']:
                analysis['security_rights_distribution'][right] = \
                    analysis['security_rights_distribution'].get(right, 0) + 1
        
        # Champs de métadonnées
        for key in doc['field_metadata'].keys():
            analysis['field_metadata_keys'][key] = \
                analysis['field_metadata_keys'].get(key, 0) + 1
        
        # Types de PJ (si définis dans metadata)
        doc_metadata = doc['metadata']
        if 'attachments' in doc_metadata:
            for att_name, att_info in doc_metadata['attachments'].items():
                att_type = att_info.get('type', 'unknown')
                analysis['attachment_types'][att_type] = \
                    analysis['attachment_types'].get(att_type, 0) + 1
    
    return analysis

# Test de gestion des métadonnées
print(f"📊 Gestion des métadonnées:")
print("=" * 60)

if client_dir.exists():
    # Lister les documents avec métadonnées
    documents = list_documents_with_metadata(CLIENT_ID)
    
    if documents:
        print(f"📚 Documents trouvés: {len(documents)}")
        print()
        
        # Afficher quelques exemples
        for i, doc in enumerate(documents[:3], 1):
            print(f"{i}. 📄 {doc['title']}")
            print(f"   📋 ID: {doc['doc_id'][:8]}...")
            print(f"   📎 {doc['attachments_count']} PJ: {', '.join(doc['attachments_list'][:3])}")
            
            if doc['security_rights']:
                rights_str = ", ".join(doc['security_rights'])
                print(f"   🔐 Droits: {rights_str}")
            else:
                print(f"   🌐 Accès libre")
            
            if doc['field_metadata']:
                print(f"   📊 Métadonnées: {', '.join(doc['field_metadata'].keys())}")
            
            print()
        
        if len(documents) > 3:
            print(f"... et {len(documents) - 3} autres documents")
        
        # Analyse statistique
        print(f"\n📊 Analyse des métadonnées:")
        analysis = analyze_metadata_distribution(CLIENT_ID)
        
        print(f"   📄 Total documents: {analysis['total_documents']}")
        print(f"   ✅ Avec métadonnées: {analysis['with_metadata']}")
        print(f"   ❌ Sans métadonnées: {analysis['without_metadata']}")
        print(f"   🔐 Avec droits: {analysis['with_security_rights']}")
        
        if analysis['security_rights_distribution']:
            print(f"   🔑 Répartition droits:")
            for right, count in analysis['security_rights_distribution'].items():
                print(f"      • {right}: {count} documents")
        
        if analysis['field_metadata_keys']:
            print(f"   📊 Champs métadonnées populaires:")
            sorted_fields = sorted(analysis['field_metadata_keys'].items(), 
                                 key=lambda x: x[1], reverse=True)
            for field, count in sorted_fields[:5]:
                print(f"      • {field}: {count} documents")
        
        # Exemple de mise à jour
        print(f"\n💡 Exemple de mise à jour des métadonnées:")
        print(f"⚠️ Décommentez pour tester:")
        print()
        
        # DÉCOMMENTEZ POUR TESTER LA MISE À JOUR:
        # sample_doc = documents[0]
        # new_metadata = {
        #     "securityRights": ["finance", "direction", "admin"],
        #     "titreDocument": "Document Mis à Jour",
        #     "fieldMetadata": {
        #         "departement": "Finance",
        #         "auteur": "Système de Test", 
        #         "date_modification": datetime.now().strftime("%Y-%m-%d"),
        #         "version": "2.0"
        #     },
        #     "attachments": sample_doc['metadata'].get('attachments', {})
        # }
        # 
        # result = update_document_metadata(CLIENT_ID, sample_doc['doc_id'], new_metadata)
        # print(f"📊 {result['message']}")
        # if result['status'] == 'success':
        #     print(f"⚠️ Document nécessite une réindexation pour appliquer les changements")
        
        print(f"💡 Utilisation:")
        print(f"   update_document_metadata('{CLIENT_ID}', doc_id, new_metadata)")
    else:
        print(f"❌ Aucun document trouvé")
        print(f"💡 Créez des documents avec create_test_document_structure()")
else:
    print(f"❌ Client {CLIENT_ID} inexistant")
    print(f"💡 Créez d'abord la structure avec prepare_client_structure()")

## 📋 Résumé - Gestion Documents Multi-PJ

✅ **Architecture Document + PJ Multiples implémentée :**

### 🏗️ **Structure Client**
- `prepare_client_structure()` - Créer l'arborescence complète
- **clients-data/[client]/documents/[uuid-36-chars]/*** - Documents avec PJ multiples
- **metadata.json** - Métadonnées centralisées par client
- **.index_state.json** - État pour synchronisation incrémentale

### 🔄 **Synchronisation Incrémentale**
- `detect_document_changes()` - Détection par signature document
- `sync_client_incremental()` - Synchronisation efficace
- **Changements détectés** : nouveaux, modifiés, supprimés
- **Performance** : Seuls les documents modifiés sont réindexés

### 📁 **Indexation Multi-PJ**
- `index_document_with_attachments()` - Indexation complète d'un document
- **Contexte préservé** : Lien document ↔ PJ dans chaque chunk
- **Métadonnées enrichies** : titreDocument, securityRights, fieldMetadata
- **Support hétérogène** : PDF, DOCX, TXT, Excel, PowerPoint

### 🔍 **Recherche Contextuelle**
- `search_with_document_context()` - Recherche avec regroupement par document
- `display_search_results_grouped()` - Affichage structuré
- **Regroupement intelligent** : Document → PJ → Chunks
- **Métadonnées visibles** : Auteur, département, version, droits

### 📊 **Gestion Métadonnées**
- `update_document_metadata()` - API de mise à jour métadonnées
- `analyze_metadata_distribution()` - Analyse statistique
- **Évolutives** : Ajout/suppression de champs sans impact
- **Centralisées** : Un fichier JSON par client

### 🔐 **Système de Droits**
- **Niveau document** : Les droits s'appliquent à toutes les PJ
- **Optionnels** : Pas de droits = accessible à tous
- **Flexibles** : `["finance", "user:marie.doe", "projet:alpha"]`
- **Filtrage automatique** : Recherche respecte les droits utilisateur

### 💾 **État Persistant**
- **Signatures document** : Hash des PJ + métadonnées
- **Synchronisation** : Détection précise des changements
- **Reprise** : Continue après interruption
- **Logs** : Historique des opérations

### 🎯 **Cas d'Usage Couverts**
- **40 clients** × **milliers de documents** ✅
- **Métadonnées évolutives** plusieurs fois/jour ✅
- **Documents multi-PJ** avec UUID ✅
- **Synchronisation incrémentale** efficace ✅
- **API de gestion** métadonnées ✅

### 🚀 **Workflow de Production**

**1. Initialisation :**
```python
prepare_client_structure("client-abc")
# → Structure prête pour documents UUID
```

**2. Ajout de documents :**
```
clients-data/client-abc/documents/
├── 123e4567-e89b-12d3-a456-426614174000/
│   ├── rapport.pdf
│   ├── annexe.xlsx  
│   └── presentation.pptx
└── metadata.json  # Métadonnées centralisées
```

**3. Synchronisation quotidienne :**
```python
sync_client_incremental("client-abc")
# → Seuls les documents modifiés sont réindexés
```

**4. Recherche contextualisée :**
```python
results = search_with_document_context("client-abc", "budget", ["finance"])
# → Résultats groupés par document avec toutes les PJ
```

**5. Mise à jour métadonnées :**
```python
update_document_metadata("client-abc", doc_uuid, new_metadata)
# → Réindexation automatique au prochain sync
```

🎉 **Le système est maintenant adapté à votre architecture document + PJ multiples et prêt pour la production !**